# MiniLAr + CNN: your turn

Work alone or in pairs for **30 minutes**. You have seen the workflow in 01/02; now assemble it for a CNN. Keep 02 open as a reference, but use a fresh kernel here. Setup, data loading, plotting, and the inference helper are supplied. The four numbered tasks are yours; `missing(...)` intentionally blocks execution until you replace it.

Open the [workshop JupyterHub](https://dune-tech.rice.edu/jupyterhub), select the **physics-ml** tile, and use that kernel. Open this notebook from `tutorial-ml/`. CPU works; CUDA is selected when available. You do not need variables from the previous notebook.

**Deliverables:** a model that passes the shape check, a working optimizer step, measured accuracy, and a row-normalized confusion matrix. Be ready to name one important confusion and explain whether your comparison with the MLP is fair. A high accuracy is not the completion criterion.

| Target time | Milestone |
| --- | --- |
| 0–3 min | Run setup; predict which local patterns distinguish particles |
| 3–11 min | Task 1: CNN and shape check |
| 11–18 min | Task 2: optimizer step and short training run |
| 18–23 min | Task 3: inference and accuracy |
| 23–30 min | Task 4: confusion matrix and interpretation |

If stuck for two minutes, open a hint or ask a neighbor/instructor. Do not spend the whole session debugging one shape. Stop optional experiments when the final discussion starts.


In [ ]:
# Import drawing tools, set style
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

mpl.rcParams['figure.figsize'] = [8, 6]
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.grid'] = True

# Import numpy and torch, set default device based on GPU availability
import numpy as np
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Set the seed to always get the same result from RNG calls in this notebook
SEED = 12345
np.random.seed(SEED)    # Setting the seed for reproducibility
torch.manual_seed(SEED) # This is how you do for torch!

def missing(prompt):
    """Pause at a workshop fill-in; replace the call with your answer."""
    raise NotImplementedError(prompt)


## Setup: reuse the same data

Keep the default split and cache from 02. Four loader workers suit the Linux Hub; set `num_workers=0` if your session has resource problems. The API calls the held-out split `test`; repeated model comparisons use it as validation data. This is one species label per cropped image, not full event reconstruction.

Before running: predict one class pair that might be hard to distinguish from these small images.


In [ ]:
from dataset.lartpc_particles import LArTPCParticles

# Specify where you want the dataset to land
LOCAL_DATA_DIR = 'dataset/minilar-dataset'

# Hosted workshop dataset (about 151 MiB; downloaded only once).
# You can also use a local path, e.g. DATA_URL='/path/to/lartpc_particles.pt'
DATA_URL = 'https://s3df.slac.stanford.edu/data/neutrino/dune-tech-analysis/minilar/raw/lartpc_particles.pt'

# Use the local MiniLAr dataset wrapper
train_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=True,
    download=DATA_URL is not None,
    url=DATA_URL,
)

train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=64,
                                           shuffle=True,
                                           num_workers=4)

# Use the local MiniLAr dataset wrapper
test_dataset = LArTPCParticles(
    LOCAL_DATA_DIR,
    train=False,
    download=DATA_URL is not None,
    url=DATA_URL,
)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=64,
                                          shuffle=False,
                                          num_workers=4)

## Task 1: build the CNN (8 minutes)

Define three convolution/activation/pooling blocks, then a linear classifier. All convolutions use a 3×3 kernel, stride 1, and padding 1. Use `LeakyReLU` after each convolution. The layer types and forward pass are supplied; fill the dimensions of the remaining blocks and classifier.

| Stage | Shape excluding batch dimension |
| --- | --- |
| Input | `[1, 28, 28]` |
| Block 1: convolution, activation, 2×2 pool | `[F, 14, 14]` |
| Block 2: convolution, activation, 2×2 pool | `[2F, 7, 7]` |
| Block 3: convolution, activation, 7×7 pool | `[4F, 1, 1]` |
| Flatten, then linear classifier | `[5]` logits |

Keep the attribute names and layer ordering: they also allow you to load the optional reference checkpoint later. Do not put softmax inside the model.

<details><summary>Hint: tensor dimensions</summary>

A convolution's input channels must equal the previous layer's output channels. `torch.flatten(features, start_dim=1)` keeps the batch dimension. `Linear` needs the flattened feature count, not the original image size.

</details>


**Syntax reference (supplied, not to memorize):**

- `torch.nn.Conv2d(in_channels, out_channels, kernel_size, padding=...)`
- `torch.nn.MaxPool2d(kernel_size, stride)` reduces each spatial dimension by the stride here.
- `torch.nn.Linear(in_features, out_features)` maps feature counts.
- `torch.flatten(features, start_dim=1)` preserves the batch dimension.

The layer types and forward-pass calls are filled in below. Your job is to make their channel counts, pooling sizes, and classifier dimensions agree with the shape table. `num_filters` is `F`. All the gaps in this task take numbers or expressions in `num_filters`.


In [ ]:
NUM_CLASSES = 5

class CNN(torch.nn.Module):
    def __init__(self, num_filters=16):
        super().__init__()
        self._feature_extractor = torch.nn.Sequential(
            torch.nn.Conv2d(1, num_filters, 3, padding=1),
            torch.nn.LeakyReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(num_filters, missing("Task 1: block 2 output channels"), 3, padding=1),
            torch.nn.LeakyReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Conv2d(missing("Task 1: block 3 input channels"),
                            missing("Task 1: block 3 output channels"), 3, padding=1),
            torch.nn.LeakyReLU(),
            torch.nn.MaxPool2d(missing("Task 1: kernel to pool a 7x7 map to 1x1"), 7),
        )
        self._classifier = torch.nn.Linear(missing("Task 1: features after flattening [4F, 1, 1]"),
                                           missing("Task 1: one output per particle class"))

    def forward(self, x):
        features = self._feature_extractor(x)
        flattened = torch.flatten(features, start_dim=1)
        return self._classifier(flattened)


In [ ]:
# Check shape and differentiability before spending time training.
shape_model = CNN(16).to(device)
probe = torch.randn(4, 1, 28, 28, device=device)
probe_logits = shape_model(probe)
assert probe_logits.shape == (4, NUM_CLASSES), 'Keep batch dimension; output five logits.'
assert torch.isfinite(probe_logits).all()
probe_logits.sum().backward()
assert all(p.grad is not None for p in shape_model.parameters()), 'Use all model layers.'
print('Shape check passed:', tuple(probe_logits.shape))


## Task 2: implement one training step (7 minutes including the run)

Complete `train_step`. It receives a model, a batch already on the correct device, a loss function, and an optimizer. Compute logits and loss, clear old gradients, backpropagate, update parameters, and return the scalar loss as a Python float. The supplied loop will call your function repeatedly.

<details><summary>Hint: which object does what?</summary>

The model produces logits, the criterion compares them with labels, the loss tensor computes gradients, and the optimizer owns the parameter update. Look back at the single-step example in 01. Do not create a new optimizer inside `train_step`.

</details>


**Syntax bank:** all operations have already appeared in 01/02. Use the following lines in a valid order, replacing `PREDICTIONS` and `TARGETS` with the variables from this function. No new API name is needed.

```python
optimizer.step()                     # update from stored gradients
loss = criterion(PREDICTIONS, TARGETS)  # compare logits with known classes
return loss.item()                   # return a Python number
optimizer.zero_grad()               # clear gradients from previous steps
logits = model(images)               # produce class scores
loss.backward()                     # compute gradients
```

The bank is out of order. Follow the dependencies: what must exist before computing the loss, and what must be computed before updating parameters?


In [ ]:
def train_step(model, images, labels, criterion, optimizer):
    # Write the forward/loss/gradient/update sequence here.
    return missing("Task 2: implement one optimizer step and return a float loss")


In [ ]:
# Check an update on a throwaway model; training below starts from scratch.
check_model = CNN(16).to(device)
check_optimizer = torch.optim.Adam(check_model.parameters(), lr=0.001)
check_criterion = torch.nn.CrossEntropyLoss()
check_images = torch.randn(4, 1, 28, 28, device=device)
check_labels = torch.tensor([0, 1, 2, 3], device=device)
before = [p.detach().clone() for p in check_model.parameters()]
check_loss = train_step(check_model, check_images, check_labels,
                        check_criterion, check_optimizer)
assert isinstance(check_loss, float) and np.isfinite(check_loss)
assert any(not torch.equal(a, b) for a, b in zip(before, check_model.parameters())),     'No parameters changed: check backward() and step().'
print('Update check passed; loss:', check_loss)


In [ ]:
# Supplied loop: your train_step does the learning.
import time
import pandas as pd
from tqdm.auto import tqdm

def train_short_run(model, loader, steps=200, max_seconds=60):
    model.train()
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = []
    if len(loader) == 0:
        raise ValueError('Training loader must contain at least one batch.')
    start = time.perf_counter()
    with tqdm(total=steps) as progress:
        while len(history) < steps:
            for images, labels in loader:
                loss = train_step(model, images.to(device), labels.to(device), criterion, optimizer)
                history.append(loss)
                progress.update(1)
                # Check the budget between batches; it cannot interrupt a slow batch.
                if len(history) >= steps or time.perf_counter() - start >= max_seconds:
                    return np.asarray(history)

# This short run is for learning the workflow, not reproducing the reference accuracy.
NUM_ITERATIONS = 200
MAX_TRAIN_SECONDS = 60
torch.manual_seed(SEED)
model = CNN(16).to(device)
loss_history = train_short_run(model, train_loader, NUM_ITERATIONS, MAX_TRAIN_SECONDS)
print('Completed optimizer steps:', len(loss_history))
plt.plot(np.arange(1, len(loss_history)+1), loss_history, alpha=0.4, label='batch loss')
plt.plot(np.arange(1, len(loss_history)+1), pd.Series(loss_history).rolling(10, min_periods=1).mean(),
         label='10-step mean')
plt.xlabel('optimizer step'); plt.ylabel('training loss'); plt.legend();


### Optional reference weights

First complete the update check and short training run. Leave the switch below `False` to evaluate **your** model. If time is short or you want to inspect the supplied long-trained model, set it to `True`. You still have to implement inference/metrics and the confusion matrix. The reference weights replace your parameters; the loss plot above remains the history of your short run.


In [ ]:
USE_REFERENCE_WEIGHTS = False
if USE_REFERENCE_WEIGHTS:
    model.load_state_dict(torch.load('weights/cnn_model.ckpt', map_location=device, weights_only=True))
result_source = 'bundled CNN checkpoint' if USE_REFERENCE_WEIGHTS else f'own CNN: {len(loss_history)} steps'
print('Evaluating:', result_source)


## Task 3: get predictions and accuracy (5 minutes)

The inference helper from 02 is provided below. Call it on your model and the **held-out** loader. Convert the `[N, 5]` scores to one class per image and compute the fraction correct. Print the result; do not use cached CNN predictions.

<details><summary>Hint: probabilities versus labels</summary>

The largest score determines the class. Reduce across the class dimension, not across examples. Accuracy compares those class indices with the integer labels.

</details>


**Call patterns from 02:** `run_test(model, loader, device=device)` returns `(labels, scores)`; `np.argmax(scores, axis=...)` selects an index; `np.mean(preds == labels)` measures accuracy. The loader names available here are `train_loader` and `test_loader`. Fill in the data split, axis, and comparison; all function names are supplied.


In [ ]:
def run_test(model, loader, device=None):
    """Runs the trained model on test data.

    Parameters
    ----------
    model : torch.nn.Module
        Model definition
    loader : torch.DataLoader
        Method to load the test data
    device : str, optional
        Device on which to put the data/model if one uses a GPU

    Returns
    -------
    numpy.ndarray
        (N,) Array of labels for each image in the test set
    numpy.ndarray
        (N, 5) Softmax scores predicted for each of the 5 particle classes
    """
    model.eval()

    # Initialize the output lists
    label_v, softmax_v = [],[]

    # Initialize the softmax function, s_i = exp(v_i)/(sum_j exp(v_j))
    softmax   = torch.nn.Softmax(dim=1)

    # Initialize a progress bar
    pbar = tqdm(total=len(loader), position=0, leave=True)

    # Loop over test data, append labels and softmax predictions
    with torch.set_grad_enabled(False):
        for data, label in loader:
            if device is not None:
                data,label = data.to(device), label.to(device)
            label_v.append  ( label.detach().reshape(-1)   )
            softmax_v.append( softmax(model(data)).detach())
            pbar.update(1)

    pbar.close()

    # Return
    return torch.concat(label_v).cpu().numpy(), torch.concat(softmax_v).cpu().numpy()

In [ ]:
labels, scores = run_test(model, missing("Task 3: training or held-out loader?"), device=device)
preds = np.argmax(scores, axis=missing("Task 3: axis containing particle classes?"))
correct = missing("Task 3: compare predicted indices to true labels")
accuracy = np.mean(correct)
print(result_source, f'held-out accuracy: {accuracy:.2%}')


In [ ]:
assert labels.shape == preds.shape == (len(test_dataset),)
assert scores.shape == (len(test_dataset), NUM_CLASSES)
assert np.allclose(scores.sum(axis=1), 1, atol=1e-5)
assert np.array_equal(preds, scores.argmax(axis=1)), 'One class per row.'
assert np.isclose(accuracy, (preds == labels).mean()), 'Accuracy is a fraction.'
assert 0 <= accuracy <= 1
print('Prediction checks passed.')


## Task 4: explain the mistakes (7 minutes)

Rows are **true classes** and columns are **predicted classes**. For each `(label, pred)` pair, decide which cell of the count matrix to increment. Then choose the sum axis that counts all predictions for each true class. This turns counts into fractions within each row; the diagonal is per-class recall.

The NumPy syntax and zero-row handling are supplied. `cm[row, column] += 1` adds one to a cell. For a 2D matrix, `sum(axis=0)` adds down rows, whereas `sum(axis=1)` adds across columns. `keepdims=True` keeps a length-one axis for broadcasting. The supplied `np.divide` writes zero wherever the row total is zero.

Fill the row index, column index, and sum axis. Pass the tiny-example check, then use the supplied plotter. This is the same matrix orientation used in 02.


In [ ]:
def make_confusion_matrix(labels, preds):
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=float)
    for label, pred in zip(labels, preds):
        row = missing("Task 4: true class or predicted class for the row?")
        column = missing("Task 4: true class or predicted class for the column?")
        cm[int(row), int(column)] += 1
    row_sums = cm.sum(axis=missing("Task 4: sum across each row — axis 0 or 1?"), keepdims=True)
    return np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums > 0)


In [ ]:
# Small example: one class-0 example is right, one is predicted as 1;
# the class-1 example is right; classes 2–4 have no examples.
tiny_cm = make_confusion_matrix(np.array([0, 0, 1]), np.array([0, 1, 1]))
expected = np.zeros((NUM_CLASSES, NUM_CLASSES))
expected[0, :2] = [0.5, 0.5]
expected[1, 1] = 1
assert tiny_cm.shape == expected.shape and np.allclose(tiny_cm, expected)
print('Confusion-matrix check passed.')


In [ ]:
def plot_confusion_matrix(cm, class_names):
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(cm, vmin=0, vmax=1, cmap='viridis')
    fig.colorbar(image, ax=ax, label='fraction of true class')
    ax.grid(False)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names)
    ax.set_xlabel('Prediction')
    ax.set_ylabel('True label')

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            value = cm[row, col]
            color = 'black' if value > 0.5 else 'white'
            ax.text(col, row, f'{value:.2f}', ha='center', va='center', color=color)

    fig.tight_layout()
    return fig, ax


cm = make_confusion_matrix(labels, preds)
plot_confusion_matrix(cm, train_dataset.class_names);


### Compare with the MLP from 02

The next cell reads only the bundled **MLP** predictions so your comparison does not depend on another notebook's live kernel. It checks that labels and ordering match. If you trained a different MLP in 02, this reference is not that model.

The bundled MLP trained for 10,000 steps. A short CNN run versus that MLP is **not a controlled architecture comparison**. Comparing both bundled checkpoints is a useful reference, but stronger conclusions need controlled budgets and repeated seeds.


In [ ]:
mlp_reference = torch.load('inference/mlp_output.pkl', map_location='cpu', weights_only=False)
assert np.array_equal(labels, mlp_reference['labels']), 'Use the same default held-out split.'
mlp_preds = mlp_reference['scores'].argmax(axis=1)
mlp_cm = make_confusion_matrix(labels, mlp_preds)
comparison = pd.DataFrame({
    'particle': train_dataset.class_names,
    'MLP reference recall': mlp_cm.diagonal(),
    'CNN recall': cm.diagonal(),
})
print('MLP reference accuracy:', np.mean(mlp_preds == labels))
print('CNN result:', result_source, accuracy)
print(comparison.to_string(index=False))


### Your conclusion (share with a neighbor)

Replace these notes with your observations:

- My CNN used **[own training / reference weights]** and **[number of steps / reference run]**.
- Its held-out accuracy was **[...]**. The class with lowest recall was **[...]**.
- One frequent confusion was **true [...] → predicted [...]**; I suspect **[...]** from the image patterns.
- Relative to the MLP, **[...]**. This comparison **[does / does not]** isolate architecture because **[...]**.

Distinguish a physical hypothesis from something you measured. You have completed the core challenge when the four checks pass and you can defend these statements.


## Optional extensions (after the core challenge)

1. Count trainable CNN parameters and compare with MLP(16). Does parameter count alone explain performance?
2. Change just the initial number of filters (8 versus 16). Start both models fresh, use equal numbers of optimizer steps and the same data split, and report accuracy and per-class recall. Do not load the 16-filter checkpoint into the 8-filter model. Keep the held-out set for validation, not repeated claims of final test performance.


In [ ]:
# Optional: your code here.
